In [1]:
pip install streamlit pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 11.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 15.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [streamlit]


In [2]:
!pip install -q streamlit pyngrok

In [3]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
import warnings
from IPython.display import display
from google.colab import files

# Read IDs as strings to keep join keys consistent
def read_csv(filename):
    df = pd.read_csv(filename, dtype="string", low_memory=False)
    df.columns = df.columns.str.strip()
    return df

balls = read_csv("ball_by_ball.csv")

# Filter immediately: retain regular innings 1 and 2 only.
if "innings" not in balls.columns:
    raise ValueError("ball_by_ball.csv is missing column: innings")

innings_number = pd.to_numeric(balls["innings"], errors="raise")

rows_before = len(balls)
balls = balls.loc[innings_number.isin([1, 2])].copy()

print(f"Excluded delivery rows: {rows_before - len(balls):,}")
print(f"Retained innings 1 and 2: {len(balls):,} rows")

# Load lookup tables only after filtering deliveries.
matches = read_csv("matches.csv")
competitions = read_csv("competitions.csv")
teams = read_csv("teams.csv")
player_mapping = read_csv("competition_team_player_mapping.csv")


def require_columns(df, required, filename):
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError(f"{filename} is missing columns: {missing}")


def make_lookup_unique(df, keys, filename):
    """
    Require exactly one row per key.

    Identical duplicate rows are collapsed.
    Conflicting duplicate rows stop the code and show examples.
    """
    duplicate_mask = df.duplicated(subset=keys, keep=False)

    if not duplicate_mask.any():
        return df.copy()

    value_columns = [col for col in df.columns if col not in keys]

    if value_columns:
        distinct_counts = (
            df.groupby(keys, dropna=False)[value_columns]
              .nunique(dropna=False)
        )
        conflicting_keys = distinct_counts.index[
            distinct_counts.gt(1).any(axis=1)
        ]

        if len(conflicting_keys) > 0:
            bad_keys = conflicting_keys.to_frame(index=False)
            print(f"Conflicting records in {filename}:")
            display(
                df.merge(bad_keys, on=keys, how="inner")
                  .sort_values(keys)
            )
            raise ValueError(
                f"{filename} has conflicting rows for {keys}. "
                "Resolve the lookup data before merging."
            )

    print(f"{filename}: collapsing identical duplicates for {keys}.")
    return df.drop_duplicates(subset=keys).copy()


# Check necessary columns
require_columns(balls, [
    "match_id",
    "batting_team_id",
    "bowling_team_id",
    "striker_id",
    "non_striker_id",
    "bowler_id",
], "ball_by_ball.csv")

require_columns(matches, ["match_id", "comp_id"], "matches.csv")
require_columns(competitions, ["comp_id"], "competitions.csv")
require_columns(teams, ["comp_id", "team_id", "team_name"], "teams.csv")
require_columns(
    player_mapping,
    ["comp_id", "team_id", "player_id", "player_name"],
    "competition_team_player_mapping.csv",
)

# Make lookup tables unique before merging
matches = make_lookup_unique(matches, ["match_id"], "matches.csv")
competitions = make_lookup_unique(
    competitions, ["comp_id"], "competitions.csv"
)
teams = make_lookup_unique(
    teams, ["comp_id", "team_id"], "teams.csv"
)
player_mapping = make_lookup_unique(
    player_mapping,
    ["comp_id", "team_id", "player_id"],
    "competition_team_player_mapping.csv",
)

original_row_count = len(balls)


# Add match columns, preserving any same-named ball-by-ball columns
match_columns = ["match_id"] + [
    col for col in matches.columns
    if col != "match_id" and col not in balls.columns
]
balls = balls.merge(
    matches[match_columns],
    on="match_id",
    how="left",
    validate="many_to_one",
)


# Add competition columns, preserving existing columns
competition_columns = ["comp_id"] + [
    col for col in competitions.columns
    if col != "comp_id" and col not in balls.columns
]
balls = balls.merge(
    competitions[competition_columns],
    on="comp_id",
    how="left",
    validate="many_to_one",
)


# Add team names using comp_id + team_id
team_lookup = teams[["comp_id", "team_id", "team_name"]].copy()

batting_teams = team_lookup.rename(columns={
    "team_id": "batting_team_id",
    "team_name": "batting_team_name",
})
balls = balls.merge(
    batting_teams,
    on=["comp_id", "batting_team_id"],
    how="left",
    validate="many_to_one",
)

bowling_teams = team_lookup.rename(columns={
    "team_id": "bowling_team_id",
    "team_name": "bowling_team_name",
})
balls = balls.merge(
    bowling_teams,
    on=["comp_id", "bowling_team_id"],
    how="left",
    validate="many_to_one",
)


# Add player mapping names for striker, non-striker, and bowler.
# These are separate from the original *_name columns in ball_by_ball.csv.
player_lookup = player_mapping[
    ["comp_id", "team_id", "player_id", "player_name"]
].copy()

def add_mapped_player_name(data, role, team_column, player_column):
    lookup = player_lookup.rename(columns={
        "team_id": f"{role}_lookup_team_id",
        "player_id": f"{role}_lookup_player_id",
        "player_name": f"{role}_mapped_name",
    })

    return data.merge(
        lookup,
        how="left",
        left_on=["comp_id", team_column, player_column],
        right_on=[
            "comp_id",
            f"{role}_lookup_team_id",
            f"{role}_lookup_player_id",
        ],
        validate="many_to_one",
    )

for role, team_column, player_column in [
    ("striker", "batting_team_id", "striker_id"),
    ("non_striker", "batting_team_id", "non_striker_id"),
    ("bowler", "bowling_team_id", "bowler_id"),
]:
    balls = add_mapped_player_name(
        balls, role, team_column, player_column
    )


# Confirm that joins did not duplicate or remove delivery rows
if len(balls) != original_row_count:
    raise RuntimeError(
        f"Row count changed from {original_row_count:,} to {len(balls):,}."
    )

output_filename = "merged_bbb.csv"
balls.to_csv(output_filename, index=False, encoding="utf-8-sig")
df = balls.copy()
print(f"Created {output_filename}")
print(f"Delivery rows: {len(balls):,}")
print(f"Columns: {len(balls.columns):,}")

files.download(output_filename)

# Convert calculation columns to numeric; treat missing values as zero
columns = [
    "innings",
    "overid",
    "runs",
    "extras1_runs",
    "extras_2_runs",
]
for col in columns:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

# Keep only first-innings deliveries
inn1 = df[df["innings"] == 1].copy()

# Runs scored per delivery
inn1["delivery_runs"] = (
    inn1["runs"] + inn1["extras1_runs"] + inn1["extras_2_runs"]
)

# Count deliveries where both extras columns are zero
inn1["no_extras_delivery"] = (
    (inn1["extras1_runs"] == 0)
    & (inn1["extras_2_runs"] == 0)
).astype(int)

# Aggregate across all matches, separately for each overid
over_mean = (
    inn1.groupby("overid", as_index=False)
    .agg(
        runs_sum=("delivery_runs", "sum"),
        no_extras_delivery_count=("no_extras_delivery", "sum"),
    )
)

# Divide runs by the count; return NaN where the count is zero
over_mean["mean_runs_per_ball_by_overid"] = (
    over_mean["runs_sum"]
    / over_mean["no_extras_delivery_count"].replace(0, np.nan)
)

# Assign each over's mean to its first-innings rows
over_mean_map = over_mean.set_index("overid")[
    "mean_runs_per_ball_by_overid"
]

df["mean_runs_per_ball_by_overid"] = np.nan

inn1_mask = df["innings"] == 1
df.loc[inn1_mask, "mean_runs_per_ball_by_overid"] = (
    df.loc[inn1_mask, "overid"].map(over_mean_map)
)

# Display the over-level summary
display(over_mean)

# Convert calculation columns to numeric; treat missing values as zero
columns = [
    "innings",
    "overid",
    "runs",
    "extras1_runs",
    "extras_2_runs",
]
for col in columns:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

# Keep only first-innings deliveries
inn1 = df[df["innings"] == 1].copy()

# Runs scored per delivery, using the specified runs and extras columns
inn1["delivery_runs"] = (
    inn1["runs"] + inn1["extras1_runs"] + inn1["extras_2_runs"]
)

# Count deliveries where both extras columns are zero
inn1["no_extras_delivery"] = (
    (inn1["extras1_runs"] == 0) &
    (inn1["extras_2_runs"] == 0)
).astype(int)

# Aggregate across all matches, separately for each venue_id
venue_mean = (
    inn1.groupby("venue_id", as_index=False)
        .agg(
            runs_sum=("delivery_runs", "sum"),
            no_extras_delivery_count=("no_extras_delivery", "sum"),
        )
)

# Divide the sum by the count; return NaN where the count is zero
venue_mean["mean_runs_per_ball_by_venueid"] = np.where(
    venue_mean["no_extras_delivery_count"] > 0,
    venue_mean["runs_sum"] / venue_mean["no_extras_delivery_count"],
    np.nan,
)

# Assign each venue's mean to its first-innings rows
venue_mean_map = venue_mean.set_index("venue_id")["mean_runs_per_ball_by_venueid"]
df["mean_runs_per_ball"] = np.nan

inn1_mask = df["innings"] == 1
df.loc[inn1_mask, "mean_runs_per_ball_by_venueid"] = (
    df.loc[inn1_mask, "venue_id"].map(venue_mean_map)
)

# Display and optionally save the venue-level summary
display(venue_mean)

df["overid_pressure"] = pd.NA

innings_2 = df["innings"] == 2
df.loc[innings_2, "overid_pressure"] = (
    pd.to_numeric(df.loc[innings_2, "overid"], errors="coerce") * 0.05
).clip(upper=1.0)

top_five = [
    "India",
    "England",
    "Australia",
    "South Africa",
    "New Zealand",
]

ranked_six_to_ten = [
    "Pakistan",
    "West Indies",
    "Bangladesh",
    "Sri Lanka",
    "Afghanistan",
]

asia_cups_scored_075 = {
    "asia cup 2022",
    "asia cup 2025",
}

def contains_team(comp_name, team):
    """Check for a case-insensitive team-name match within the competition name."""
    pattern = rf"(?<!\w){re.escape(team)}(?!\w)"
    return bool(re.search(pattern, comp_name, flags=re.IGNORECASE))

def assign_comp_diff(row):
    comp_name = str(row["comp_name"]).strip()
    match_format = str(row["match_format"]).strip().upper()

    # Tri competitions are capped at 0.75:
    # 0.75 if a top-five team is named; otherwise 0.5.
    if re.search(r"Tri", comp_name, flags=re.IGNORECASE):
        if any(contains_team(comp_name, team) for team in top_five):
            return 0.75
        return 0.5

    # Rating 1.0: competition name contains IPL
    if re.search(r"IPL", comp_name, flags=re.IGNORECASE):
        return 1.0

    # Rating 1.0: T20I featuring at least two distinct top-five nations
    if match_format == "T20I":
        top_five_found = {
            team for team in top_five
            if contains_team(comp_name, team)
        }
        if len(top_five_found) >= 2:
            return 1.0

    # Rating 0.75: competition name contains Asia Cup 2022 or Asia Cup 2025
    comp_name_folded = comp_name.casefold()
    if any(asia_cup in comp_name_folded for asia_cup in asia_cups_scored_075):
        return 0.75

    # Rating 0.75: competition name features a team ranked 6–10
    if any(contains_team(comp_name, team) for team in ranked_six_to_ten):
        return 0.75

    # Rating 0.5: all remaining competitions
    return 0.5

# Apply to the DataFrame containing comp_name and match_format
df["comp_diff"] = df.apply(assign_comp_diff, axis=1)

def calculate_recent_ratings(match_dates, half_life_days=365):
    """
    Exponential time-decay recency rating.
    Latest valid date in the dataset is the reference date (age 0).
    """
    if not np.isfinite(half_life_days) or half_life_days <= 0:
        raise ValueError("half_life_days must be positive and finite.")

    dates = pd.to_datetime(match_dates, errors="coerce", utc=True)
    reference_date = dates.max()
    if pd.isna(reference_date):
        raise ValueError("No valid match_date values were found.")

    age_days = (reference_date - dates).dt.total_seconds() / 86400
    ratings = pd.Series(
        np.power(0.5, age_days / half_life_days),
        index=match_dates.index,
        dtype="float64",
    )
    ratings.loc[dates.isna()] = np.nan
    return ratings, reference_date


df["recent_ratings"], recency_reference_date = calculate_recent_ratings(
    df["match_date"], half_life_days=365
)
print(f"Recency reference date (latest dataset match): {recency_reference_date.date()}")

def calculate_cricket_metrics(data):
    """
    Calculate metrics using the existing dataframe.

    Dataset-specific rules
    ----------------------
    Legal extras types:
        blank, none, nan, byes, legbyes, penalty

    Illegal extras types:
        wide, overstep, other

    Bowler runs conceded:
        Batter runs + extras1_runs only when extras_type is
        wide, overstep, or other.

        extras_2_runs is NEVER charged to the bowler.

    Bowler-credited dismissals:
        Bowled, Caught, Hit Wicket, LBW, Stump

    Non-bowler dismissals:
        Obstracting the field, Obstructing the field,
        Retire Out, RunOut

    Existing columns and row order are preserved.
    No files are loaded or exported.
    """

    b = data.copy()

    # ------------------------------------------------------------
    # 1. Validate required columns
    # ------------------------------------------------------------
    extra2_col = next(
        (
            col
            for col in ["extras_2_runs", "extras2_runs"]
            if col in b.columns
        ),
        None,
    )

    if extra2_col is None:
        raise ValueError(
            "Missing secondary-extras column: "
            "expected extras_2_runs or extras2_runs."
        )

    required_columns = [
        "match_id",
        "match_date",
        "innings",
        "batting_team_id",
        "bowling_team_id",
        "striker_id",
        "bowler_id",
        "dismissed_player_id",
        "runs",
        "extras1_runs",
        extra2_col,
        "extras_type",
        "dismissal_type",
        "is_four",
        "is_six",
        "pace_or_spin",
    ]

    missing = [
        col for col in required_columns
        if col not in b.columns
    ]

    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    # ------------------------------------------------------------
    # 2. Helper functions
    # ------------------------------------------------------------
    def normalise(series):
        """Ignore capitalisation, spaces, and punctuation."""
        return (
            series.astype("string")
            .fillna("")
            .str.strip()
            .str.lower()
            .str.replace(r"[^a-z0-9]", "", regex=True)
        )

    def numeric(column):
        return pd.to_numeric(
            b[column], errors="raise"
        ).fillna(0)

    def boolean_flag(column):
        values = (
            b[column]
            .astype("string")
            .fillna("")
            .str.strip()
            .str.lower()
        )

        true_values = {"true", "1", "1.0"}
        false_values = {
            "false", "0", "0.0", "", "none", "nan"
        }

        unknown = (
            set(values.unique())
            - true_values
            - false_values
        )

        if unknown:
            raise ValueError(
                f"Unexpected values in {column}: {sorted(unknown)}"
            )

        return values.isin(true_values).astype(int)

    def safe_divide(numerator, denominator, multiplier=1):
        """Return NaN when the denominator is zero."""
        return (
            numerator.div(denominator.replace(0, np.nan))
            * multiplier
        )

    def aggregate(frame, keys, specification):
        return (
            frame.groupby(
                keys,
                dropna=False,
                observed=True,
                sort=False,
            )
            .agg(**specification)
            .reset_index()
        )

    def overs_notation(legal_balls):
        """Display 20 legal balls as '3.2', not decimal overs."""
        balls = legal_balls.astype("int64")

        return (
            (balls // 6).astype(str)
            + "."
            + (balls % 6).astype(str)
        )

    # ------------------------------------------------------------
    # 3. Runs and extras classification
    # ------------------------------------------------------------
    batter_runs = numeric("runs")
    extras1 = numeric("extras1_runs")
    extras2 = numeric(extra2_col)

    if (
        batter_runs.lt(0).any()
        or extras1.lt(0).any()
        or extras2.lt(0).any()
    ):
        raise ValueError(
            "Negative runs require explicit handling."
        )

    extras_type = normalise(b["extras_type"])

    legal_extra_types = {
        "",
        "none",
        "nan",
        "byes",
        "legbyes",
        "penalty",
    }

    illegal_extra_types = {
        "wide",
        "overstep",
        "other",
    }

    unknown_extras = (
        set(extras_type.unique())
        - legal_extra_types
        - illegal_extra_types
    )

    if unknown_extras:
        raise ValueError(
            "Unrecognised extras_type values: "
            f"{sorted(unknown_extras)}"
        )

    b["calc_batter_runs"] = batter_runs

    # Both extras columns contribute to the team total.
    b["calc_team_runs"] = (
        batter_runs + extras1 + extras2
    )

    b["calc_legal_ball"] = (
        extras_type.isin(legal_extra_types).astype(int)
    )

    # Retains the previous convention: exclude wides only.
    b["calc_ball_faced"] = (
        extras_type.ne("wide").astype(int)
    )

    b["calc_scoring_shot"] = (
        batter_runs.gt(0)
        & b["calc_ball_faced"].eq(1)
    ).astype(int)

    b["calc_four"] = boolean_flag("is_four")
    b["calc_six"] = boolean_flag("is_six")

    b["calc_boundary_runs"] = (
        b["calc_four"] * 4
        + b["calc_six"] * 6
    )

    # Dot ball = legal delivery with zero total team runs.
    b["calc_dot_ball"] = (
        b["calc_legal_ball"].eq(1)
        & b["calc_team_runs"].eq(0)
    ).astype(int)

    # ------------------------------------------------------------
    # 4. Bowler runs conceded
    # ------------------------------------------------------------
    # Only extras1_runs can be charged to the bowler,
    # and only for these three extras types.
    bowler_chargeable = extras_type.isin(
        illegal_extra_types
    )

    b["calc_bowler_extras"] = extras1.where(
        bowler_chargeable, 0
    )

    # extras2 is deliberately excluded.
    b["calc_bowler_runs_conceded"] = (
        batter_runs + b["calc_bowler_extras"]
    )

    # ------------------------------------------------------------
    # 5. Dismissals — only labels from your screenshot
    # ------------------------------------------------------------
    dismissal = normalise(b["dismissal_type"])

    bowler_dismissals = {
        "bowled",       # Bowled
        "caught",       # Caught
        "hitwicket",    # Hit Wicket
        "lbw",          # LBW
        "stump",        # Stump
    }

    non_bowler_dismissals = {
        "obstractingthefield",  # Obstracting the field
        "obstructingthefield",  # Obstructing the field
        "retireout",            # Retire Out
        "runout",               # RunOut
    }

    # Missing dismissal values mean no dismissal.
    non_dismissals = {
        "",
        "none",
        "nan",
    }

    unknown_dismissals = (
        set(dismissal.unique())
        - bowler_dismissals
        - non_bowler_dismissals
        - non_dismissals
    )

    if unknown_dismissals:
        raise ValueError(
            "Unexpected dismissal types: "
            f"{sorted(unknown_dismissals)}"
        )

    # Both dismissal groups count towards team wickets
    # and the dismissed batter's dismissal count.
    actual_dismissal = dismissal.isin(
        bowler_dismissals | non_bowler_dismissals
    )

    b["calc_team_wicket"] = actual_dismissal.astype(int)

    # Only the five specified types credit the bowler.
    b["calc_bowler_wicket"] = (
        dismissal.isin(bowler_dismissals).astype(int)
    )

    # Normalise numeric IDs for summary joins.
    # Original ID columns remain unchanged.
    striker_key = pd.to_numeric(
        b["striker_id"],
        errors="raise",
    ).astype("Int64")

    dismissed_key = pd.to_numeric(
        b["dismissed_player_id"].where(
            actual_dismissal,
            pd.NA,
        ),
        errors="raise",
    ).astype("Int64")

    invalid_dismissed_id = actual_dismissal & (
        dismissed_key.isna()
        | dismissed_key.eq(0)
    )

    if invalid_dismissed_id.any():
        raise ValueError(
            "A dismissal has a missing or zero "
            "dismissed_player_id."
        )

    # Temporary keys are kept separate from returned delivery data.
    summary_data = b.copy()
    summary_data["_striker_key"] = striker_key
    summary_data["_dismissed_key"] = dismissed_key

    innings_keys = [
        "match_id",
        "match_date",
        "innings",
    ]

    # ------------------------------------------------------------
    # 6. Batting summaries
    # ------------------------------------------------------------
    def batting_summary(context):
        team_keys = context + ["batting_team_id"]

        batting = aggregate(
            summary_data,
            team_keys + ["_striker_key"],
            {
                "batter_runs": ("calc_batter_runs", "sum"),
                "balls_faced": ("calc_ball_faced", "sum"),
                "scoring_shots": ("calc_scoring_shot", "sum"),
                "fours": ("calc_four", "sum"),
                "sixes": ("calc_six", "sum"),
                "boundary_runs": ("calc_boundary_runs", "sum"),
            },
        ).rename(
            columns={"_striker_key": "player_id"}
        )

        dismissed_rows = summary_data.loc[
            summary_data["calc_team_wicket"].eq(1)
        ]

        dismissals = aggregate(
            dismissed_rows,
            team_keys + ["_dismissed_key"],
            {
                "dismissals": ("calc_team_wicket", "sum"),
            },
        ).rename(
            columns={"_dismissed_key": "player_id"}
        )

        # Attribute dismissals to dismissed_player_id, not striker_id.
        # This also retains dismissed non-strikers with no balls faced.
        result = batting.merge(
            dismissals,
            on=team_keys + ["player_id"],
            how="outer",
            validate="one_to_one",
        )

        count_columns = [
            "batter_runs",
            "balls_faced",
            "scoring_shots",
            "fours",
            "sixes",
            "boundary_runs",
            "dismissals",
        ]

        result[count_columns] = (
            result[count_columns].fillna(0)
        )

        result["batting_average"] = safe_divide(
            result["batter_runs"],
            result["dismissals"],
        )

        result["strike_rate"] = safe_divide(
            result["batter_runs"],
            result["balls_faced"],
            100,
        )

        result["scoring_shot_pct"] = safe_divide(
            result["scoring_shots"],
            result["balls_faced"],
            100,
        )

        result["boundary_runs_pct"] = safe_divide(
            result["boundary_runs"],
            result["batter_runs"],
            100,
        )

        return result

    # ------------------------------------------------------------
    # 7. Bowling summaries
    # ------------------------------------------------------------
    def bowling_summary(keys):
        result = aggregate(
            b,
            keys,
            {
                "physical_deliveries": (
                    "calc_legal_ball", "size"
                ),
                "legal_balls": (
                    "calc_legal_ball", "sum"
                ),
                "runs_conceded": (
                    "calc_bowler_runs_conceded", "sum"
                ),
                "bowler_wickets": (
                    "calc_bowler_wicket", "sum"
                ),
                "dot_balls": (
                    "calc_dot_ball", "sum"
                ),
            },
        )

        result["overs"] = overs_notation(
            result["legal_balls"]
        )

        result["decimal_overs"] = (
            result["legal_balls"] / 6
        )

        result["economy"] = safe_divide(
            result["runs_conceded"],
            result["legal_balls"],
            6,
        )

        result["bowling_strike_rate"] = safe_divide(
            result["legal_balls"],
            result["bowler_wickets"],
        )

        result["dot_ball_pct"] = safe_divide(
            result["dot_balls"],
            result["legal_balls"],
            100,
        )

        return result

    # ------------------------------------------------------------
    # 8. Innings summary
    # ------------------------------------------------------------
    innings_summary = aggregate(
        b,
        innings_keys + [
            "batting_team_id",
            "bowling_team_id",
        ],
        {
            "team_runs": (
                "calc_team_runs", "sum"
            ),
            "team_wickets": (
                "calc_team_wicket", "sum"
            ),
            "bowler_credited_wickets": (
                "calc_bowler_wicket", "sum"
            ),
            "legal_balls": (
                "calc_legal_ball", "sum"
            ),
            "physical_deliveries": (
                "calc_legal_ball", "size"
            ),
        },
    )

    innings_summary["overs"] = overs_notation(
        innings_summary["legal_balls"]
    )

    innings_summary["run_rate"] = safe_divide(
        innings_summary["team_runs"],
        innings_summary["legal_balls"],
        6,
    )

    # ------------------------------------------------------------
    # 9. Return results
    # ------------------------------------------------------------
    return {
        "deliveries": b,

        "innings_summary": innings_summary,

        "batting_by_innings": batting_summary(
            innings_keys
        ),

        "batting_by_match": batting_summary(
            ["match_id", "match_date"]
        ),

        # Overall batting totals separated by team.
        "batting_summary": batting_summary([]),

        "bowling_by_innings": bowling_summary(
            innings_keys + [
                "bowling_team_id",
                "bowler_id",
            ]
        ),

        "bowling_by_match": bowling_summary(
            [
                "match_id",
                "match_date",
                "bowling_team_id",
                "bowler_id",
            ]
        ),

        # Overall bowling totals separated by team.
        "bowling_summary": bowling_summary(
            ["bowling_team_id", "bowler_id"]
        ),

        "bowling_by_pace_spin": bowling_summary(
            ["pace_or_spin"]
        ),

        "bowling_by_innings_pace_spin": bowling_summary(
            innings_keys + [
                "bowling_team_id",
                "pace_or_spin",
            ]
        ),
    }


# ================================================================
# APPLY TO YOUR EXISTING df
# ================================================================
metrics = calculate_cricket_metrics(df)

# Preserve original columns, including existing chase-pressure
# columns, and add/update the calculated delivery-level columns.
df = metrics["deliveries"]

innings_summary = metrics["innings_summary"]

batting_by_innings = metrics["batting_by_innings"]
batting_by_match = metrics["batting_by_match"]
batting_summary = metrics["batting_summary"]

bowling_by_innings = metrics["bowling_by_innings"]
bowling_by_match = metrics["bowling_by_match"]
bowling_summary = metrics["bowling_summary"]

bowling_by_pace_spin = metrics["bowling_by_pace_spin"]
bowling_by_innings_pace_spin = metrics[
    "bowling_by_innings_pace_spin"
]

print(f"Delivery rows retained: {len(df):,}")

print("\nInnings summary")
display(innings_summary.head())

print("\nBatting summary")
display(batting_summary.head())

print("\nBowling summary")
display(bowling_summary.head())

print("\nPace / Spin summary")
display(bowling_by_pace_spin)

def add_chase_wickets_importance(data):
    """
    Adds:
        chase_pressure  : innings 2 only
        wickets_left    : both innings, before the delivery
        match_importance: stage-based, both innings

    Uses existing calc_legal_ball and calc_team_wicket columns.

    Requirements:
    - Complete innings, not player/phase-filtered data.
    - Rows in chronological order within each innings.

    Dataset convention:
    - first_inn_revised_overs and second_inn_revised_overs
      contain legal-ball allocations, e.g. 114 = 19 overs.
    - Missing/zero allocations default to 120 balls.
    - Positive revised_target overrides first-innings total + 1.

    Match-level revised targets/allocations are applied throughout
    the chase; this does not reconstruct in-play revision timing.
    """

    required = [
        "match_id",
        "innings",
        "runs",
        "extras1_runs",
        "extras_2_runs",
        "calc_legal_ball",
        "calc_team_wicket",
        "first_inn_revised_overs",
        "second_inn_revised_overs",
        "revised_target",
        "stage",
    ]

    missing = [col for col in required if col not in data.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    original_index = data.index.copy()
    b = data.copy().reset_index(drop=True)

    def numeric(series):
        """Parse numbers without silently ignoring invalid text."""
        text = series.astype("string").str.strip()
        empty = text.isna() | text.str.lower().isin(
            {"", "none", "nan"}
        )
        return pd.to_numeric(
            text.mask(empty, pd.NA),
            errors="raise",
        ).astype("float64")

    innings = numeric(b["innings"])
    innings1 = innings.eq(1)
    innings2 = innings.eq(2)

    if not innings.isin([1, 2]).all():
        raise ValueError(
            "This calculation supports innings 1 and 2 only. "
            "Handle super overs separately."
        )

    # ------------------------------------------------------------
    # 1. Match metadata
    # ------------------------------------------------------------
    def match_numeric(column):
        values = numeric(b[column])

        # Missing/zero means no revision supplied.
        values = values.where(values.gt(0))

        grouped = values.groupby(b["match_id"], sort=False)

        if grouped.nunique().gt(1).any():
            raise ValueError(
                f"Conflicting {column} values within a match."
            )

        # Populate all delivery rows from the match-level value.
        return grouped.transform("first")

    revised_target = match_numeric("revised_target")
    first_allocation = match_numeric(
        "first_inn_revised_overs"
    ).fillna(120)
    second_allocation = match_numeric(
        "second_inn_revised_overs"
    ).fillna(120)

    if (
        first_allocation.mod(1).ne(0).any()
        or second_allocation.mod(1).ne(0).any()
    ):
        raise ValueError(
            "Revised allocations must be whole legal-ball counts. "
            "Check the units in the source data."
        )

    b["innings_ball_allocation"] = np.where(
        innings1,
        first_allocation,
        second_allocation,
    )

    # ------------------------------------------------------------
    # 2. Pre-delivery runs, wickets and legal balls
    # ------------------------------------------------------------
    delivery_runs = sum(
        numeric(b[col]).fillna(0)
        for col in ["runs", "extras1_runs", "extras_2_runs"]
    )

    legal_ball = numeric(b["calc_legal_ball"])
    team_wicket = numeric(b["calc_team_wicket"])

    for name, values in [
        ("calc_legal_ball", legal_ball),
        ("calc_team_wicket", team_wicket),
    ]:
        if not values.isin([0, 1]).all():
            raise ValueError(f"{name} must contain only 0 or 1.")

    group_keys = [b["match_id"], innings]

    b["runs_before_ball"] = (
        delivery_runs.groupby(group_keys, sort=False).cumsum()
        - delivery_runs
    )

    b["legal_balls_before_ball"] = (
        legal_ball.groupby(group_keys, sort=False).cumsum()
        - legal_ball
    )

    b["wickets_lost_before_ball"] = (
        team_wicket.groupby(group_keys, sort=False).cumsum()
        - team_wicket
    )

    if (
        b["wickets_lost_before_ball"].gt(10).any()
        or b["wickets_lost_before_ball"].lt(0).any()
    ):
        raise ValueError(
            "Invalid wicket totals. Check duplicate deliveries."
        )

    # Both innings; resets separately for each match and innings.
    b["wickets_left"] = (
        10 - b["wickets_lost_before_ball"]
    ).astype("Int64")

    # ------------------------------------------------------------
    # 3. First-innings score and target
    # ------------------------------------------------------------
    first_totals = (
        delivery_runs.loc[innings1]
        .groupby(b.loc[innings1, "match_id"], sort=False)
        .sum()
    )

    actual_first_score = b["match_id"].map(first_totals)

    # Revised target overrides the actual first-innings score.
    # Effective first-innings score = revised target - 1.
    b["first_innings_effective_score"] = (
        revised_target.sub(1).combine_first(actual_first_score)
    )

    target = b["first_innings_effective_score"] + 1

    if target.loc[innings2].isna().any():
        raise ValueError(
            "Some chases have neither a first-innings total "
            "nor a revised target."
        )

    b["chase_target"] = target.where(innings2)

    # ------------------------------------------------------------
    # 4. Initial and current required run rate
    # ------------------------------------------------------------
    b["initial_required_run_rate"] = (
        target * 6 / second_allocation
    ).where(innings2)

    b["runs_required"] = (
        target - b["runs_before_ball"]
    ).clip(lower=0).where(innings2)

    b["balls_remaining"] = (
        second_allocation - b["legal_balls_before_ball"]
    ).where(innings2)

    # Zero/negative remaining balls have no defined run rate.
    valid_remaining = b["balls_remaining"].where(
        b["balls_remaining"].gt(0)
    )

    b["current_required_run_rate"] = (
        b["runs_required"] * 6 / valid_remaining
    )

    # ------------------------------------------------------------
    # 5. Chase pressure
    # ------------------------------------------------------------
    b["chase_pressure"] = (
        b["current_required_run_rate"]
        / b["initial_required_run_rate"].where(
            b["initial_required_run_rate"].gt(0)
        )
    ).where(innings2)

    # ------------------------------------------------------------
    # 6. Match importance — your exact weights
    # ------------------------------------------------------------
    stage_key = (
        b["stage"]
        .astype("string")
        .fillna("")
        .str.strip()
        .str.lower()
        .str.replace(r"[\s_-]+", "", regex=True)
    )

    importance_weights = {
        "final": 1.0,
        "eliminator": 0.75,
        "qualifier": 0.75,
        "semifinal": 0.75,
        "super4": 0.5,
        "others": 0.25,
        "league": 0.25,
    }

    unknown_stage = ~stage_key.isin(importance_weights)

    if unknown_stage.any():
        raise ValueError(
            "Unexpected stage values: "
            f"{b.loc[unknown_stage, 'stage'].unique().tolist()}"
        )

    b["match_importance"] = stage_key.map(
        importance_weights
    ).astype(float)

    # The same match must have the same importance in both innings.
    if (
        b.groupby("match_id")["match_importance"]
        .nunique()
        .gt(1)
        .any()
    ):
        raise ValueError(
            "Conflicting stage weights within a match."
        )

    b.index = original_index
    return b


# ================================================================
# APPLY TO EXISTING COMPLETE DELIVERY DATA
# ================================================================
df = add_chase_wickets_importance(df)

# Update existing metrics dictionary, if present.
if "metrics" in globals() and isinstance(metrics, dict):
    metrics["deliveries"] = df

preview_columns = [
    "match_id",
    "innings",
    "innings_ball_allocation",
    "first_innings_effective_score",
    "chase_target",
    "runs_before_ball",
    "runs_required",
    "balls_remaining",
    "initial_required_run_rate",
    "current_required_run_rate",
    "chase_pressure",
    "wickets_left",
    "stage",
    "match_importance",
]

display(df[preview_columns].head(10))

print("\nSecond-innings preview")
display(
    df.loc[
        pd.to_numeric(df["innings"]).eq(2),
        preview_columns,
    ].head(15)
)

def evaluate_pressure_ratings(data, prior_balls=600, recency_half_life_days=913):
    """
    Calculate batter and bowler pressure ratings.

    Includes:
        - Date-based exponential recency decay (latest dataset date is reference).
        - Separate batting/bowling chase contexts using relative escalation
          and absolute current required run rate.
        - Population smoothing controlled by prior_balls.
        - Batting-position weights for bowler-credited wickets.
        - Automatic dismissed-player batting-position lookup.

    Requires an existing delivery dataframe with the calculation
    and context columns listed below.

    batting_order must describe the striker's recorded batting
    position for that match and innings, not the wicket number.

    Returns:
        batter_pressure_ratings
        bowler_pressure_ratings
    """

    if not np.isfinite(prior_balls) or prior_balls <= 0:
        raise ValueError("prior_balls must be a positive finite number.")

    # ============================================================
    # 1. Validate input columns and select innings 1 and 2
    # ============================================================
    required = [
        "match_id",
        "match_date",
        "innings",
        "batting_team_id",
        "striker_id",
        "non_striker_id",
        "dismissed_player_id",
        "bowler_id",
        "batting_order",
        "runs",
        "calc_ball_faced",
        "calc_team_wicket",
        "calc_bowler_runs_conceded",
        "calc_bowler_wicket",
        "calc_legal_ball",
        "mean_runs_per_ball_by_venueid",
        "mean_runs_per_ball_by_overid",
        "overid_pressure",
        "comp_diff",
        "chase_pressure",
        "current_required_run_rate",
        "wickets_left",
        "match_importance",
    ]

    missing = [col for col in required if col not in data.columns]

    if missing:
        raise ValueError(
            f"Missing required columns: {missing}. "
            "Run your earlier calculation and context cells first."
        )

    innings = pd.to_numeric(data["innings"], errors="coerce")

    b = (
        data.loc[innings.isin([1, 2])]
        .copy()
        .reset_index(drop=True)
    )

    if b.empty:
        raise ValueError("No innings 1 or 2 deliveries found.")

    b["innings"] = pd.to_numeric(b["innings"], errors="raise")

    inn1 = b["innings"].eq(1)
    inn2 = b["innings"].eq(2)

    # Dataset-relative time decay: newest match_date is day zero.
    parsed_dates = pd.to_datetime(b["match_date"], errors="coerce", utc=True)
    recency_reference_date = parsed_dates.max()
    if pd.isna(recency_reference_date):
        raise ValueError("No valid match_date values were found.")
    if not np.isfinite(recency_half_life_days) or recency_half_life_days <= 0:
        raise ValueError("recency_half_life_days must be positive and finite.")
    age_days = (recency_reference_date - parsed_dates).dt.total_seconds() / 86400
    b["recent_ratings"] = np.power(0.5, age_days / recency_half_life_days)
    b.loc[parsed_dates.isna(), "recent_ratings"] = np.nan

    # Columns that must be numeric and complete on every delivery.
    complete_numeric_columns = [
        "runs",
        "calc_ball_faced",
        "calc_team_wicket",
        "calc_bowler_runs_conceded",
        "calc_bowler_wicket",
        "calc_legal_ball",
        "comp_diff",
        "recent_ratings",
        "wickets_left",
        "match_importance",
    ]

    # These may be missing where they are not applicable.
    context_columns = [
        "mean_runs_per_ball_by_venueid",
        "mean_runs_per_ball_by_overid",
        "overid_pressure",
        "chase_pressure",
        "current_required_run_rate",
    ]

    for col in complete_numeric_columns + context_columns:
        b[col] = pd.to_numeric(b[col], errors="coerce")

    for col in complete_numeric_columns:
        if not np.isfinite(b[col].to_numpy(dtype=float)).all():
            raise ValueError(
                f"{col} contains missing, invalid or infinite values."
            )

    for mask, columns in [
        (
            inn1,
            [
                "mean_runs_per_ball_by_venueid",
                "mean_runs_per_ball_by_overid",
            ],
        ),
        (
            inn2,
            ["overid_pressure", "chase_pressure", "current_required_run_rate"],
        ),
    ]:
        for col in columns:
            values = b.loc[mask, col].to_numpy(dtype=float)

            if not np.isfinite(values).all():
                raise ValueError(
                    f"{col} contains invalid values in the innings "
                    "where it is required."
                )

    binary_columns = [
        "calc_ball_faced",
        "calc_team_wicket",
        "calc_bowler_wicket",
        "calc_legal_ball",
    ]

    for col in binary_columns:
        if not b[col].isin([0, 1]).all():
            raise ValueError(f"{col} must contain only 0 or 1.")

    for col in ["runs", "calc_bowler_runs_conceded"]:
        if b[col].lt(0).any():
            raise ValueError(f"{col} cannot be negative.")

    for col in ["chase_pressure", "current_required_run_rate"]:
        if b.loc[inn2, col].lt(0).any():
            raise ValueError(f"{col} cannot be negative.")

    for col in ["comp_diff", "recent_ratings", "match_importance"]:
        if not (b[col].gt(0) & b[col].le(1)).all():
            raise ValueError(
                f"{col} must be greater than 0 and at most 1."
            )

    if not b["wickets_left"].between(0, 10).all():
        raise ValueError("wickets_left must be between 0 and 10.")

    if (
        b["calc_bowler_wicket"] > b["calc_team_wicket"]
    ).any():
        raise ValueError("A bowler wicket must also be a team wicket.")

    # ============================================================
    # 2. Normalize player IDs
    # ============================================================
    def clean_ids(series):
        cleaned = (
            series.astype("string")
            .str.strip()
            .str.replace(r"^(\d+)\.0+$", r"\1", regex=True)
        )

        invalid = cleaned.str.lower().isin(
            ["", "none", "nan", "<na>", "0", "-1"]
        )

        return cleaned.mask(invalid)

    striker = clean_ids(b["striker_id"])
    non_striker = clean_ids(b["non_striker_id"])
    bowler = clean_ids(b["bowler_id"])
    dismissed = clean_ids(b["dismissed_player_id"])

    if striker.isna().any() or bowler.isna().any():
        raise ValueError("Missing or invalid striker_id or bowler_id.")

    team_wicket_mask = b["calc_team_wicket"].eq(1)
    bowler_wicket_mask = b["calc_bowler_wicket"].eq(1)

    # Ignore placeholder dismissal IDs on non-wicket deliveries.
    dismissed = dismissed.where(team_wicket_mask)

    if dismissed.loc[team_wicket_mask].isna().any():
        raise ValueError(
            "A team wicket has no valid dismissed_player_id."
        )

    valid_dismissed = (
        dismissed.eq(striker) | dismissed.eq(non_striker)
    )

    if not valid_dismissed.loc[team_wicket_mask].fillna(False).all():
        raise ValueError(
            "A dismissed player is neither the striker nor non-striker."
        )

    # ============================================================
    # 3. Create dismissed_batting_position
    # ============================================================
    # Build a lookup using each striker's recorded batting_order.
    # Match within the same match, innings and batting team.
    innings_keys = ["match_id", "innings", "batting_team_id"]
    lookup_keys = innings_keys + ["_player_id"]

    if b[innings_keys].isna().any().any():
        raise ValueError(
            "match_id, innings and batting_team_id cannot be missing."
        )

    position_lookup = b[innings_keys].copy()
    position_lookup["_player_id"] = striker
    position_lookup["_batting_position"] = pd.to_numeric(
        b["batting_order"],
        errors="coerce",
    )

    position_lookup = position_lookup.loc[
        position_lookup["_player_id"].notna()
        & position_lookup["_batting_position"].isin(range(1, 12))
    ].copy()

    position_counts = (
        position_lookup.groupby(lookup_keys)["_batting_position"]
        .nunique()
    )

    conflicts = position_counts[position_counts.gt(1)]

    if not conflicts.empty:
        raise ValueError(
            f"Found {len(conflicts)} player-innings with conflicting "
            "batting_order values. Correct these before rating.\n"
            f"Examples:\n{conflicts.head()}"
        )

    position_lookup = position_lookup.drop_duplicates(
        subset=lookup_keys
    )

    dismissed_lookup = b[innings_keys].copy()
    dismissed_lookup["_player_id"] = dismissed
    dismissed_lookup["_row_order"] = np.arange(len(b))

    mapped_positions = (
        dismissed_lookup.merge(
            position_lookup,
            on=lookup_keys,
            how="left",
            sort=False,
            validate="many_to_one",
        )
        .sort_values("_row_order")
    )

    b["dismissed_batting_position"] = (
        mapped_positions["_batting_position"].to_numpy()
    )

    missing_position_mask = (
        bowler_wicket_mask
        & b["dismissed_batting_position"].isna()
    )

    if missing_position_mask.any():
        examples = b.loc[
            missing_position_mask,
            [
                "match_id",
                "innings",
                "batting_team_id",
                "striker_id",
                "dismissed_player_id",
                "batting_order",
            ],
        ].head(10)

        raise ValueError(
            f"{int(missing_position_mask.sum())} bowler-credited "
            "wickets could not be matched to a batting position. "
            "Check batting_order and whether the innings data "
            "contains the dismissed player's deliveries.\n"
            f"Examples:\n{examples.to_string(index=False)}"
        )

    wicket_position_weights = {
        1: 1.00,
        2: 1.00,
        3: 1.00,
        4: 0.85,
        5: 0.70,
        6: 0.60,
        7: 0.50,
        8: 0.40,
        9: 0.30,
        10: 0.20,
        11: 0.10,
    }

    b["wicket_position_weight"] = (
        b["dismissed_batting_position"]
        .map(wicket_position_weights)
        .where(bowler_wicket_mask, 0.0)
    )

    b["position_weighted_wicket"] = (
        b["calc_bowler_wicket"]
        * b["wicket_position_weight"]
    )

    # ============================================================
    # 4. Percentile helper
    # ============================================================
    def percentile_01(values, higher_is_better=True):
        values = pd.Series(values, copy=True, dtype=float)
        valid = values.notna()

        result = pd.Series(
            np.nan,
            index=values.index,
            dtype=float,
        )

        if not valid.any():
            return result

        ranks = values.loc[valid].rank(
            method="average",
            ascending=higher_is_better,
        )

        span = ranks.max() - ranks.min()

        if span == 0:
            result.loc[valid] = 0.5
        else:
            result.loc[valid] = (
                (ranks - ranks.min()) / span
            )

        return result

    # ============================================================
    # 5. Delivery-context weights
    # ============================================================
    # Percentile direction is deliberately consistent:
    #   - Higher venue/over scoring baselines = greater bowler pressure.
    #   - Higher absolute RRR = greater batter pressure.
    #   - Higher chase escalation (current RRR / initial RRR)
    #     = greater batter pressure.
    #   - More wickets lost = greater batting pressure.
    #   - More wickets remaining = greater bowling pressure.
    #
    # percentile_01() is higher-is-better by default, so the
    # complementary (1 - percentile) is used for the opposite role.

    venue_ease = percentile_01(
        b.loc[inn1, "mean_runs_per_ball_by_venueid"]
    )

    over_ease = percentile_01(
        b.loc[inn1, "mean_runs_per_ball_by_overid"]
    )

    wicket_pressure = 1 - b["wickets_left"] / 10
    bowling_wickets_resource = b["wickets_left"] / 10

    batting_context = pd.Series(0.0, index=b.index)
    bowling_context = pd.Series(0.0, index=b.index)

    # First innings:
    # Lower scoring baselines favour batting-pressure context;
    # higher scoring baselines favour bowling-pressure context.
    batting_context.loc[inn1] = (
        (1 - venue_ease)
        + (1 - over_ease)
        + wicket_pressure.loc[inn1]
    ) / 3

    bowling_context.loc[inn1] = (
        venue_ease
        + over_ease
        + bowling_wickets_resource.loc[inn1]
    ) / 3

    # Second innings: relative chase escalation and absolute RRR
    # are distinct pressure signals and both are retained.
    relative_chase_percentile = percentile_01(
        b.loc[inn2, "chase_pressure"]
    )
    absolute_rr_percentile = percentile_01(
        b.loc[inn2, "current_required_run_rate"]
    )

    # Batting: higher chase escalation, higher absolute RRR,
    # and fewer wickets remaining (more wickets lost).
    batting_context.loc[inn2] = (
        relative_chase_percentile
        + absolute_rr_percentile
        + wicket_pressure.loc[inn2]
    ) / 3

    # Bowling: lower chase escalation and lower absolute RRR
    # put the bowler under more pressure; more wickets in hand
    # give the batting side resources and make defending harder.
    bowling_context.loc[inn2] = (
        (1 - relative_chase_percentile)
        + (1 - absolute_rr_percentile)
        + bowling_wickets_resource.loc[inn2]
    ) / 3

    base_weight = (
        b["comp_diff"]
        * b["recent_ratings"]
        * b["match_importance"]
    )

    batting_weight = base_weight * (0.5 + batting_context)
    bowling_weight = base_weight * (0.5 + bowling_context)

    # ============================================================
    # 6. Batter totals
    # ============================================================
    batter_ids = pd.Index(
        pd.concat(
            [striker, non_striker, dismissed],
            ignore_index=True,
        ).dropna().unique(),
        name="batter_id",
    )

    batting = pd.DataFrame(index=batter_ids)

    batting["runs_scored"] = b["runs"].groupby(striker).sum()

    batting["balls_faced"] = (
        b["calc_ball_faced"].groupby(striker).sum()
    )

    batting["dismissals"] = dismissed.value_counts()

    batting["weighted_runs"] = (
        b["runs"] * batting_weight
    ).groupby(striker).sum()

    batting["weighted_balls"] = (
        b["calc_ball_faced"] * batting_weight
    ).groupby(striker).sum()

    # Includes dismissed non-strikers.
    batting["weighted_dismissals"] = (
        b["calc_team_wicket"] * batting_weight
    ).groupby(dismissed).sum()

    batting = batting.fillna(0)

    batting["batting_average"] = (
        batting["runs_scored"]
        / batting["dismissals"].replace(0, np.nan)
    )

    batting["strike_rate"] = (
        100 * batting["runs_scored"]
        / batting["balls_faced"].replace(0, np.nan)
    )

    batting["pressure_batting_average"] = (
        batting["weighted_runs"]
        / batting["weighted_dismissals"].replace(0, np.nan)
    )

    batting["pressure_strike_rate"] = (
        100 * batting["weighted_runs"]
        / batting["weighted_balls"].replace(0, np.nan)
    )

    # ============================================================
    # 7. Bowler totals with batting-position wicket weights
    # ============================================================
    bowling_events = pd.DataFrame({
        "bowler_id": bowler,

        # Traditional statistics remain unchanged.
        "runs_conceded": b["calc_bowler_runs_conceded"],
        "wickets": b["calc_bowler_wicket"],
        "legal_balls": b["calc_legal_ball"],

        # Position-only wicket value.
        "position_weighted_wickets": (
            b["position_weighted_wicket"]
        ),

        # Delivery-pressure-weighted runs and balls.
        "weighted_runs": (
            b["calc_bowler_runs_conceded"] * bowling_weight
        ),
        "weighted_balls": (
            b["calc_legal_ball"] * bowling_weight
        ),

        # Delivery pressure multiplied by wicket-position value.
        "weighted_wickets": (
            b["position_weighted_wicket"] * bowling_weight
        ),
    })

    bowling = bowling_events.groupby("bowler_id").sum()

    bowling["bowling_average"] = (
        bowling["runs_conceded"]
        / bowling["wickets"].replace(0, np.nan)
    )

    bowling["economy_rate"] = (
        6 * bowling["runs_conceded"]
        / bowling["legal_balls"].replace(0, np.nan)
    )

    bowling["mean_wicket_position_weight"] = (
        bowling["position_weighted_wickets"]
        / bowling["wickets"].replace(0, np.nan)
    )

    bowling["pressure_bowling_average"] = (
        bowling["weighted_runs"]
        / bowling["weighted_wickets"].replace(0, np.nan)
    )

    bowling["pressure_economy_rate"] = (
        6 * bowling["weighted_runs"]
        / bowling["weighted_balls"].replace(0, np.nan)
    )

    # ============================================================
    # 8. Small-sample smoothing
    # ============================================================
    bat_actual_balls = batting["balls_faced"].sum()
    bowl_actual_balls = bowling["legal_balls"].sum()

    bat_weighted_balls = batting["weighted_balls"].sum()
    bowl_weighted_balls = bowling["weighted_balls"].sum()

    if min(
        bat_actual_balls,
        bowl_actual_balls,
        bat_weighted_balls,
        bowl_weighted_balls,
    ) <= 0:
        raise ValueError("Insufficient balls to calculate ratings.")

    if (
        batting["weighted_dismissals"].sum() <= 0
        or bowling["weighted_wickets"].sum() <= 0
    ):
        raise ValueError(
            "At least one batter dismissal and one bowler-credited "
            "wicket are required for population baselines."
        )

    # Convert prior_balls into each role's weighted-ball units.
    bat_prior = (
        prior_balls * bat_weighted_balls / bat_actual_balls
    )

    bowl_prior = (
        prior_balls * bowl_weighted_balls / bowl_actual_balls
    )

    bat_run_rate = (
        batting["weighted_runs"].sum() / bat_weighted_balls
    )

    bat_dismissal_rate = (
        batting["weighted_dismissals"].sum()
        / bat_weighted_balls
    )

    bowl_run_rate = (
        bowling["weighted_runs"].sum() / bowl_weighted_balls
    )

    # Uses the same position-adjusted wicket definition as players.
    bowl_wicket_rate = (
        bowling["weighted_wickets"].sum()
        / bowl_weighted_balls
    )

    if bowl_run_rate <= 0:
        raise ValueError(
            "Population runs conceded must be positive."
        )

    batting["rating_batting_average"] = (
        batting["weighted_runs"] + bat_prior * bat_run_rate
    ) / (
        batting["weighted_dismissals"]
        + bat_prior * bat_dismissal_rate
    )

    batting["rating_strike_rate"] = 100 * (
        batting["weighted_runs"] + bat_prior * bat_run_rate
    ) / (
        batting["weighted_balls"] + bat_prior
    )

    bowling["rating_bowling_average"] = (
        bowling["weighted_runs"] + bowl_prior * bowl_run_rate
    ) / (
        bowling["weighted_wickets"]
        + bowl_prior * bowl_wicket_rate
    )

    bowling["rating_economy_rate"] = 6 * (
        bowling["weighted_runs"] + bowl_prior * bowl_run_rate
    ) / (
        bowling["weighted_balls"] + bowl_prior
    )

    # ============================================================
    # 9. Combine metrics and scale to 0-100
    # ============================================================
    batting["combined_score"] = (
        0.40 * percentile_01(
            batting["rating_batting_average"]
        )
        + 0.60 * percentile_01(
            batting["rating_strike_rate"]
        )
    )

    bowling["combined_score"] = (
        0.50 * percentile_01(
            bowling["rating_bowling_average"],
            higher_is_better=False,
        )
        + 0.50 * percentile_01(
            bowling["rating_economy_rate"],
            higher_is_better=False,
        )
    )

    # ============================================================
    # 9A. Identify provisional players
    # ============================================================
    batting["provisional"] = (
        batting["balls_faced"].lt(prior_balls)
    )

    bowling["provisional"] = (
        bowling["legal_balls"].lt(prior_balls)
    )

    # ============================================================
    # 9B. Reduce provisional == true combined scores by 99.99%
    # ============================================================
    batting.loc[
        batting["provisional"],
        "combined_score"
    ] *= 0.0001

    bowling.loc[
        bowling["provisional"],
        "combined_score"
    ] *= 0.0001

    # ============================================================
    # 9C. Calculate final pressure ratings
    # ============================================================
    batting["pressure_rating"] = (
        100 * percentile_01(batting["combined_score"])
    )

    bowling["pressure_rating"] = (
        100 * percentile_01(bowling["combined_score"])
    )

    # ============================================================
    # 10. Add existing player names, when available
    # ============================================================
    def add_names(summary, role, id_values):
        for column in [
            f"{role}_mapped_name",
            f"{role}_name",
        ]:
            if column not in b.columns:
                continue

            names = pd.DataFrame({
                "player_id": id_values,
                "player_name": (
                    b[column].astype("string").str.strip()
                ),
            })

            names = names.loc[
                names["player_id"].notna()
                & names["player_name"].notna()
                & names["player_name"].ne("")
            ]

            lookup = (
                names.groupby(["player_id", "player_name"])
                .size()
                .rename("count")
                .reset_index()
                .sort_values("count", ascending=False)
                .drop_duplicates("player_id")
                .set_index("player_id")["player_name"]
            )

            mapped = summary.index.to_series().map(lookup)

            if "player_name" in summary.columns:
                summary["player_name"] = (
                    summary["player_name"].fillna(mapped)
                )
            else:
                summary["player_name"] = mapped

        return summary

    batting = add_names(batting, "striker", striker)
    batting = add_names(batting, "non_striker", non_striker)
    bowling = add_names(bowling, "bowler", bowler)

    # ============================================================
    # 11. Format final outputs
    # ============================================================
    def finish(summary, id_column, exposure_column):
        result = (
            summary.reset_index()
            .sort_values(
                ["pressure_rating", exposure_column, id_column],
                ascending=[False, False, True],
            )
            .reset_index(drop=True)
        )

        result.insert(
            0,
            "rank",
            result["pressure_rating"]
            .rank(method="min", ascending=False)
            .astype(int),
        )

        first_columns = ["rank", id_column]

        if "player_name" in result.columns:
            first_columns.append("player_name")

        first_columns += ["pressure_rating", "provisional"]

        remaining = [
            col for col in result.columns
            if col not in first_columns
        ]

        return result[first_columns + remaining]

    return (
        finish(batting, "batter_id", "balls_faced"),
        finish(bowling, "bowler_id", "legal_balls"),
    )


# ================================================================
# APPLY TO YOUR EXISTING DELIVERY DATAFRAME
# ================================================================
batter_pressure_ratings, bowler_pressure_ratings = (
    evaluate_pressure_ratings(df, prior_balls=600, recency_half_life_days=913)
)

Excluded delivery rows: 38
Retained innings 1 and 2: 200,961 rows
Created merged_bbb.csv
Delivery rows: 200,961
Columns: 68


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,overid,runs_sum,no_extras_delivery_count,mean_runs_per_ball_by_overid
0,1,5949,5099,1.166699
1,2,6994,5075,1.378128
2,3,7734,5071,1.525143
3,4,7770,5083,1.528625
4,5,8027,5087,1.577944
5,6,7977,5067,1.574304
6,7,6380,5093,1.2527
7,8,6828,5061,1.34914
8,9,6931,5071,1.366792
9,10,7062,5058,1.396204


,venue_id,runs_sum,no_extras_delivery_count,mean_runs_per_ball_by_venueid
0,1,5518,3761,1.467163
1,10,5791,3427,1.689816
2,100,622,352,1.767045
3,101,1437,819,1.754579
4,103,294,235,1.251064
...,...,...,...,...
92,86,1652,1176,1.404762
93,89,465,357,1.302521
94,9,816,468,1.743590
95,90,2085,1490,1.399329


Recency reference date (latest dataset match): 2026-11-01
Delivery rows retained: 200,961

Innings summary


... (6 outputs left out)

,pace_or_spin,physical_deliveries,legal_balls,runs_conceded,bowler_wickets,dot_balls,overs,decimal_overs,economy,bowling_strike_rate,dot_ball_pct
0,Pace,123859,117200,178801,6501,42675,19533.2,19533.333333,9.153635,18.027996,36.412116
1,Spin,77102,75212,100710,3760,25337,12535.2,12535.333333,8.03409,20.003191,33.687443


,match_id,innings,innings_ball_allocation,first_innings_effective_score,chase_target,runs_before_ball,runs_required,balls_remaining,initial_required_run_rate,current_required_run_rate,chase_pressure,wickets_left,stage,match_importance
0,13511,1,120.0,103.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,10,League,0.25
1,13511,1,120.0,103.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,10,League,0.25
2,13511,1,120.0,103.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,10,League,0.25
3,13511,1,120.0,103.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,10,League,0.25
4,13511,1,120.0,103.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,10,League,0.25
5,13511,1,120.0,103.0,NaN,6.0,NaN,NaN,NaN,NaN,NaN,10,League,0.25
6,13511,1,120.0,103.0,NaN,6.0,NaN,NaN,NaN,NaN,NaN,9,League,0.25
7,13511,1,120.0,103.0,NaN,6.0,NaN,NaN,NaN,NaN,NaN,9,League,0.25
8,13511,1,120.0,103.0,NaN,6.0,NaN,NaN,NaN,NaN,NaN,9,League,0.25
9,13511,1,120.0,103.0,NaN,6.0,NaN,NaN,NaN,NaN,NaN,9,League,0.25



Second-innings preview


,match_id,innings,innings_ball_allocation,first_innings_effective_score,chase_target,runs_before_ball,runs_required,balls_remaining,initial_required_run_rate,current_required_run_rate,chase_pressure,wickets_left,stage,match_importance
121,13511,2,120.0,103.0,104.0,0.0,104.0,120.0,5.2,5.200000,1.000000,10,League,0.25
122,13511,2,120.0,103.0,104.0,0.0,104.0,119.0,5.2,5.243697,1.008403,10,League,0.25
123,13511,2,120.0,103.0,104.0,4.0,100.0,118.0,5.2,5.084746,0.977836,10,League,0.25
124,13511,2,120.0,103.0,104.0,4.0,100.0,117.0,5.2,5.128205,0.986193,10,League,0.25
125,13511,2,120.0,103.0,104.0,4.0,100.0,116.0,5.2,5.172414,0.994695,10,League,0.25
126,13511,2,120.0,103.0,104.0,5.0,99.0,115.0,5.2,5.165217,0.993311,10,League,0.25
127,13511,2,120.0,103.0,104.0,6.0,98.0,114.0,5.2,5.157895,0.991903,10,League,0.25
128,13511,2,120.0,103.0,104.0,10.0,94.0,113.0,5.2,4.991150,0.959837,10,League,0.25
129,13511,2,120.0,103.0,104.0,11.0,93.0,112.0,5.2,4.982143,0.958104,10,League,0.25
130,13511,2,120.0,103.0,104.0,11.0,93.0,111.0,5.2,5.027027,0.966736,10,League,0.25


In [4]:
print(
    f"BATTER PRESSURE RATINGS: "
    f"{len(batter_pressure_ratings):,} unique batters"
)
display(batter_pressure_ratings.round(2))

BATTER PRESSURE RATINGS: 893 unique batters


,rank,batter_id,player_name,pressure_rating,provisional,runs_scored,balls_faced,dismissals,weighted_runs,weighted_balls,weighted_dismissals,batting_average,strike_rate,pressure_batting_average,pressure_strike_rate,rating_batting_average,rating_strike_rate,combined_score
0,1,50284,Vaibhav Sooryavanshi,100.00,False,1509,676.0,35,291.97,127.55,5.89,43.11,223.22,49.6,228.9,39.2,197.44,1.00
1,2,5039,Rajat Patidar,99.89,False,1541,918.0,44,281.44,160.45,6.92,35.02,167.86,40.7,175.41,34.58,164.77,0.99
2,3,3647,Shreyas Iyer,99.78,False,2856,1865.0,78,403.5,249.75,10.02,36.62,153.14,40.27,161.56,35.62,156.94,0.99
3,4,5553,Phil Salt,99.66,False,2782,1643.0,76,306.86,180.80,8.46,36.61,169.32,36.29,169.73,32.36,161.55,0.99
4,5,6281,Abhishek Sharma,99.55,False,4332,2317.0,130,560.68,298.52,17.80,33.32,186.97,31.49,187.82,30.14,178.79,0.99
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
888,889,360,Bhuvneshwar Kumar,0.45,True,149,169.0,16,22.85,25.68,2.66,9.31,88.17,8.58,88.99,18.07,126.84,0.00
889,890,5944,Mohammed Siraj,0.34,True,58,72.0,9,11.28,19.80,2.04,6.44,80.56,5.52,56.96,18.01,122.28,0.00
890,891,7967,Noor Ahmad,0.22,True,92,114.0,18,17.87,21.77,3.24,5.11,80.7,5.52,82.07,15.98,126.81,0.00
891,892,3644,Kuldeep Yadav,0.11,True,157,201.0,17,20.49,28.41,2.92,9.24,78.11,7.02,72.11,17.08,120.95,0.00


In [5]:
print(
    f"BOWLER PRESSURE RATINGS: "
    f"{len(bowler_pressure_ratings):,} unique bowlers"
)
display(bowler_pressure_ratings.round(2))

BOWLER PRESSURE RATINGS: 647 unique bowlers


,rank,bowler_id,player_name,pressure_rating,provisional,runs_conceded,wickets,legal_balls,position_weighted_wickets,weighted_runs,weighted_balls,weighted_wickets,bowling_average,economy_rate,mean_wicket_position_weight,pressure_bowling_average,pressure_economy_rate,rating_bowling_average,rating_economy_rate,combined_score
0,1,2490,Mohammad Nawaz,100.00,False,1192,66,1004,52.35,135.95,121.41,7.04,18.06,7.12,0.79,19.31,6.72,24.78,7.65,1.00
1,2,5837,Mahedi Hasan,99.85,False,1074,50,914,42.50,140.35,117.33,5.80,21.48,7.05,0.85,24.21,7.18,28.55,7.94,0.99
2,3,6355,Shaheen Afridi,99.69,False,1322,65,990,49.15,185.63,147.43,7.83,20.34,8.01,0.76,23.7,7.55,27.41,8.08,0.99
3,4,4900,Wanindu Hasaranga,99.46,False,2690,133,2034,105.70,347.61,260.55,13.30,20.23,7.94,0.79,26.14,8.0,28.14,8.25,0.98
4,4,5979,Abrar Ahmed,99.46,False,851,46,764,32.55,146.36,129.99,5.46,18.5,6.68,0.71,26.78,6.76,30.27,7.64,0.98
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
642,643,3605,Aiden Markram,0.54,True,668,13,397,11.70,108.19,63.60,1.85,51.38,10.1,0.90,58.48,10.21,43.51,9.53,0.00
643,644,6162,Kulwant Khejroliya,0.39,True,124,3,56,2.40,37.17,15.51,0.33,41.33,13.29,0.80,113.29,14.38,42.27,9.85,0.00
644,645,67036,Brijesh Sharma,0.23,True,450,14,273,9.35,153.3,86.11,3.01,32.14,9.89,0.67,50.97,10.68,42.71,9.86,0.00
645,646,8221,Nitish Reddy,0.00,True,977,27,565,19.40,205.1,117.10,4.19,36.19,10.38,0.72,48.95,10.51,42.87,9.88,0.00


In [6]:
# Export pressure ratings for Streamlit dashboard
batter_pressure_ratings.to_csv(
    "batter_pressure_ratings.csv",
    index=False
)

bowler_pressure_ratings.to_csv(
    "bowler_pressure_ratings.csv",
    index=False
)

print("Pressure rating CSVs exported successfully.")

Pressure rating CSVs exported successfully.


In [7]:
%%writefile superrr_pressure_dashboard.py
# ------------------------------------------------------------
# SupeRR Selector Season 2 | Pressure Scale Dashboard
# Reads batter_pressure_ratings.csv and bowler_pressure_ratings.csv directly from the runtime
# ------------------------------------------------------------
import streamlit as st
import pandas as pd
import numpy as np
import re
from pathlib import Path
import warnings

st.set_page_config(page_title="SupeRR | Pressure Scale", page_icon="🏏", layout="wide")

st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@400;500;600;700&family=Space+Grotesk:wght@500;600;700&display=swap');
html, body, [class*="css"] { font-family: 'DM Sans', sans-serif; }
.stApp { background: radial-gradient(circle at 10% 0%, #14243b 0%, #0b1220 42%, #080d16 100%); color: #e8eef8; }
.block-container { padding-top: 2rem; max-width: 1500px; }
.hero { padding: 26px 30px; border: 1px solid #263952; border-radius: 22px; background: linear-gradient(115deg, rgba(24,42,67,.95), rgba(13,23,39,.92)); box-shadow: 0 18px 50px rgba(0,0,0,.22); }
.hero-kicker { color: #6ee7c5; font-size: 12px; font-weight: 700; letter-spacing: 2.5px; text-transform: uppercase; }
.hero-title { font-family: 'Space Grotesk'; font-size: 38px; font-weight: 700; letter-spacing: -1.4px; color: #f4f8ff; margin: 5px 0 2px; }
.hero-sub { color: #a8b8ce; font-size: 15px; }
.panel { background: rgba(17,29,47,.82); border: 1px solid #26364d; border-radius: 18px; padding: 20px 22px; }
.metric-card { background: linear-gradient(145deg,#17263b,#101a2b); border: 1px solid #2a3b55; border-radius: 16px; padding: 17px 19px; min-height: 105px; }
.metric-label { color: #91a5bf; font-size: 12px; letter-spacing: 1px; text-transform: uppercase; font-weight: 700; }
.metric-value { color: #f4f8ff; font-family: 'Space Grotesk'; font-size: 29px; font-weight: 700; margin-top: 7px; }
.metric-note { color: #7d91ad; font-size: 12px; margin-top: 2px; }
.section-title { font-family: 'Space Grotesk'; font-size: 20px; font-weight: 700; color: #eaf2ff; margin: 2px 0 12px; }
.small-note { color: #8297b4; font-size: 12px; }
div[data-testid="stSelectbox"] label, div[data-testid="stMultiSelect"] label { color: #c8d6e8 !important; font-weight: 600; }
[data-testid="stMetric"] { background: #142137; border: 1px solid #2b3d57; padding: 15px; border-radius: 14px; }
[data-testid="stMetricLabel"] { color: #9cb0ca !important; }
[data-testid="stMetricValue"] { color: #f1f6ff !important; font-family: 'Space Grotesk'; }
hr { border-color: #26364d; }
</style>
""", unsafe_allow_html=True)

CATS = ["Elite", "Good", "Satisfactory", "Unsatisfactory"]
CAT_COLORS = {"Elite":"#65e6bd", "Good":"#79b8ff", "Satisfactory":"#ffc96b", "Unsatisfactory":"#ff8293"}

def category_from_pct(x):
    if pd.isna(x):
        return "Unrated"
    if x > 90:
        return "Elite"
    if x > 80:
        return "Good"
    if x > 70:
        return "Satisfactory"
    return "Unsatisfactory"

def clean_rating_file(file_path, role):
    df = pd.read_csv(file_path)
    # Allow the canonical evaluator outputs, plus common naming variations.
    name_col = next((c for c in ["player_name", "name", "Player", "player"] if c in df.columns), None)
    id_col = "batter_id" if role == "Batting" else "bowler_id"
    if name_col is None:
        name_col = id_col if id_col in df.columns else None
    if name_col is None or "pressure_rating" not in df.columns:
        raise ValueError(f"{role} file needs a player_name (or {id_col}) column and pressure_rating.")
    df["Player"] = df[name_col].astype(str).str.strip()
    df["Pressure Percentile"] = pd.to_numeric(df["pressure_rating"], errors="coerce")

    # Minimum sample-size eligibility: fewer than 100 balls is unrated.
    sample_col = "balls_faced" if role == "Batting" else "legal_balls"
    if sample_col in df.columns:
        df[sample_col] = pd.to_numeric(df[sample_col], errors="coerce")
        low_sample = df[sample_col] < 100
        df.loc[low_sample, "Pressure Percentile"] = np.nan
    else:
        low_sample = pd.Series(False, index=df.index)

    df["Category"] = df["Pressure Percentile"].apply(category_from_pct)
    df.loc[low_sample, "Category"] = "Unsatisfactory"
    metric_map = ({"Batting Average":"batting_average", "Strike Rate":"strike_rate"} if role == "Batting"
                  else {"Bowling Average":"bowling_average", "Economy Rate":"economy_rate"})
    for display, source in metric_map.items():
        if source in df.columns:
            df[display] = pd.to_numeric(df[source], errors="coerce")
        else:
            df[display] = np.nan
    df["Role"] = role
    return df

st.markdown("""
<div class="hero">
  <div class="hero-kicker">SupeRR Selector · Season 2</div>
  <div class="hero-title">PRESSURE SCALE <span style="color:#6ee7c5">//</span> PLAYER LAB</div>
  <div class="hero-sub">Explore player pressure percentiles, raw performance metrics and category tiers.</div>
</div>
""", unsafe_allow_html=True)

with st.sidebar:
    st.markdown("### 🎛️ Filters")
    category_selection = st.multiselect("Player Category", CATS, default=[], help="Leave empty to include every category.")
    logic = st.radio("", ["OR", "AND"], horizontal=True,
                     help="When 'All' option is selected in Player Pool then OR: player qualifies in either role, AND: player must qualify in both batting and bowling.")
    role_view = st.radio("Player Pool", ["All", "Batters", "Bowlers"], index=0)

# Read both rating CSVs directly from the same runtime/environment as this app.
BAT_CSV = Path(__file__).resolve().parent / "batter_pressure_ratings.csv"
BOWL_CSV = Path(__file__).resolve().parent / "bowler_pressure_ratings.csv"

try:
    if not BAT_CSV.exists():
        raise FileNotFoundError(f"Missing file: {BAT_CSV}")
    if not BOWL_CSV.exists():
        raise FileNotFoundError(f"Missing file: {BOWL_CSV}")
    batting = clean_rating_file(BAT_CSV, "Batting")
    bowling = clean_rating_file(BOWL_CSV, "Bowling")
except Exception as e:
    st.error(f"Could not load rating CSVs: {e}")
    st.info("Place both CSV files in the same directory as superrr_pressure_dashboard.py.")
    st.stop()

all_data = pd.concat([batting, bowling], ignore_index=True, sort=False)
all_data = all_data[all_data["Player"].notna() & all_data["Player"].ne("")]

# Build player eligibility from selected categories and role pool.
players = sorted(all_data["Player"].unique().tolist(), key=str.casefold)
if role_view == "Batters": players = sorted(batting["Player"].dropna().unique().tolist(), key=str.casefold)
if role_view == "Bowlers": players = sorted(bowling["Player"].dropna().unique().tolist(), key=str.casefold)

if category_selection:
    eligible = []
    for player in players:
        pbat = batting[batting.Player == player]
        pbowl = bowling[bowling.Player == player]
        bat_match = bool(len(pbat) and pbat.iloc[0]["Category"] in category_selection)
        bowl_match = bool(len(pbowl) and pbowl.iloc[0]["Category"] in category_selection)
        if role_view == "Batters": ok = bat_match
        elif role_view == "Bowlers": ok = bowl_match
        elif logic == "OR": ok = bat_match or bowl_match
        else:
            # AND means a player must have both role records and both must be in selected category set.
            ok = bool(len(pbat) and len(pbowl) and bat_match and bowl_match)
        if ok: eligible.append(player)
    players = eligible

# Sort filtered player list by the highest relevant pressure rating.
def player_sort_score(player):
    vals=[]
    if role_view != "Bowlers":
        x=batting.loc[batting.Player==player,"Pressure Percentile"]
        if len(x): vals.extend(x.dropna().tolist())
    if role_view != "Batters":
        x=bowling.loc[bowling.Player==player,"Pressure Percentile"]
        if len(x): vals.extend(x.dropna().tolist())
    return max(vals) if vals else -1
players = sorted(players, key=lambda p: (-player_sort_score(p), p.casefold()))

col1, col2, col3 = st.columns([1.15, 1, 1])
with col1:
    st.markdown(
    '<div class="section-title">🔎 Player Selector '
    '<span style="font-size:10px; font-weight:400;">(descending order of pressure ratings)</span>'
    '</div>',
    unsafe_allow_html=True
    )
    selected_player = st.selectbox("Player (filtered & pressure-ranked)", players, index=0 if players else None,
                                   placeholder="Choose a player…", label_visibility="collapsed")
with col2:
    st.markdown('<div class="section-title">🏷️ Tier Key</div>', unsafe_allow_html=True)
    st.markdown(
    "<span style='color:#65e6bd'>● Elite >90</span> &nbsp; "
    "<span style='color:#79b8ff'>● 90≥Good>80</span><br>"
    "<span style='color:#ffc96b'>● 80≥Satisfactory>70</span> &nbsp; "
    "<span style='color:#ff8293'>● Unsatisfactory ≤70</span>",
    unsafe_allow_html=True
    )
with col3:
    st.markdown('<div class="section-title">📌 Current View</div>', unsafe_allow_html=True)
    st.markdown(f"<div class='small-note'>{len(players):,} players available · {len(category_selection) if category_selection else 'All'} category filter(s)</div>", unsafe_allow_html=True)

if not players:
    st.warning("No players match these filters. Try changing the category selection, OR/AND logic, or player pool.")
    st.stop()

st.markdown("---")
if selected_player:
    b_row = batting[batting.Player == selected_player]
    w_row = bowling[bowling.Player == selected_player]
    bat_exists, bowl_exists = bool(len(b_row)), bool(len(w_row))
    bat_row = b_row.iloc[0] if bat_exists else None
    bowl_row = w_row.iloc[0] if bowl_exists else None

    st.markdown(f"<div class='section-title'>PLAYER PROFILE <span style='color:#6ee7c5'>/</span> {selected_player}</div>", unsafe_allow_html=True)
    left, right = st.columns(2, gap="large")
    with left:
        st.markdown("<div class='panel'>", unsafe_allow_html=True)
        st.markdown("### 🏏 Batting Pressure Rating")
        if bat_exists:
            pct=bat_row["Pressure Percentile"]
            cat=bat_row["Category"]
            if pd.isna(pct):
                st.markdown(f"<div style='font-size:42px;font-family:Space Grotesk;font-weight:700;color:{CAT_COLORS.get(cat,'#eaf2ff')}'>—<span style='font-size:18px;color:#8ea3bd'> / 100</span></div><div style='font-weight:700;color:{CAT_COLORS.get(cat,'#eaf2ff')};font-size:16px'>{cat}</div>", unsafe_allow_html=True)
                st.caption("Pressure percentile undefined · fewer than 100 balls faced")
            else:
                pct=float(pct)
                st.markdown(f"<div style='font-size:42px;font-family:Space Grotesk;font-weight:700;color:{CAT_COLORS.get(cat,'#eaf2ff')}'>{pct:.2f}<span style='font-size:18px;color:#8ea3bd'> / 100</span></div><div style='font-weight:700;color:{CAT_COLORS.get(cat,'#eaf2ff')};font-size:16px'>{cat}</div>", unsafe_allow_html=True)
                st.progress(max(0,min(100,int(round(pct))))/100)
            m1,m2=st.columns(2)
            m1.metric("Batting Average", "—" if pd.isna(bat_row["Batting Average"]) else f"{bat_row['Batting Average']:.2f}", help="Raw, un-pressure-adjusted batting average")
            m2.metric("Strike Rate", "—" if pd.isna(bat_row["Strike Rate"]) else f"{bat_row['Strike Rate']:.2f}", help="Raw, un-pressure-adjusted strike rate")
        else: st.info("No batting pressure rating available for this player.")
        st.markdown("</div>", unsafe_allow_html=True)
    with right:
        st.markdown("<div class='panel'>", unsafe_allow_html=True)
        st.markdown("### 🎯 Bowling Pressure Rating")
        if bowl_exists:
            pct=bowl_row["Pressure Percentile"]
            cat=bowl_row["Category"]
            if pd.isna(pct):
                st.markdown(f"<div style='font-size:42px;font-family:Space Grotesk;font-weight:700;color:{CAT_COLORS.get(cat,'#eaf2ff')}'>—<span style='font-size:18px;color:#8ea3bd'> / 100</span></div><div style='font-weight:700;color:{CAT_COLORS.get(cat,'#eaf2ff')};font-size:16px'>{cat}</div>", unsafe_allow_html=True)
                st.caption("Pressure percentile undefined · fewer than 100 legal balls")
            else:
                pct=float(pct)
                st.markdown(f"<div style='font-size:42px;font-family:Space Grotesk;font-weight:700;color:{CAT_COLORS.get(cat,'#eaf2ff')}'>{pct:.2f}<span style='font-size:18px;color:#8ea3bd'> / 100</span></div><div style='font-weight:700;color:{CAT_COLORS.get(cat,'#eaf2ff')};font-size:16px'>{cat}</div>", unsafe_allow_html=True)
                st.progress(max(0,min(100,int(round(pct))))/100)
            m1,m2=st.columns(2)
            m1.metric("Bowling Average", "—" if pd.isna(bowl_row["Bowling Average"]) else f"{bowl_row['Bowling Average']:.2f}", help="Raw, un-pressure-adjusted bowling average; lower is better")
            m2.metric("Economy Rate", "—" if pd.isna(bowl_row["Economy Rate"]) else f"{bowl_row['Economy Rate']:.2f}", help="Raw, un-pressure-adjusted economy rate; lower is better")
        else: st.info("No bowling rating available for this player.")
        st.markdown("</div>", unsafe_allow_html=True)

    with st.expander("View the player's underlying rating row"):
        tabs=st.tabs(["Batting row", "Bowling row"])
        with tabs[0]:
            if bat_exists: st.dataframe(b_row, use_container_width=True, hide_index=True)
            else: st.write("No batting row.")
        with tabs[1]:
            if bowl_exists: st.dataframe(w_row, use_container_width=True, hide_index=True)
            else: st.write("No bowling row.")

st.markdown("---")
st.markdown('<div class="section-title">📋 Category Leaderboard</div>', unsafe_allow_html=True)

# Build leaderboard from the current filtered player pool. The sort score must
# use the pressure rating for the selected category/role—not the player's other
# role rating, which could otherwise incorrectly move them up the table.
leader_rows = []
for player in players:
    br = batting[batting["Player"] == player]
    wr = bowling[bowling["Player"] == player]
    bat = br.iloc[0] if not br.empty else None
    bowl = wr.iloc[0] if not wr.empty else None

    bat_pct = float(bat["Pressure Percentile"]) if bat is not None and pd.notna(bat["Pressure Percentile"]) else np.nan
    bowl_pct = float(bowl["Pressure Percentile"]) if bowl is not None and pd.notna(bowl["Pressure Percentile"]) else np.nan

    # Score only the role(s) relevant to Player Pool and selected category.
    sort_candidates = []
    if role_view != "Bowlers" and bat is not None:
        if not category_selection or bat["Category"] in category_selection:
            sort_candidates.append(bat_pct)
    if role_view != "Batters" and bowl is not None:
        if not category_selection or bowl["Category"] in category_selection:
            sort_candidates.append(bowl_pct)
    sort_pressure = max((v for v in sort_candidates if pd.notna(v)), default=np.nan)

    leader_rows.append({
        "Player": player,
        "Batting Category": bat["Category"] if bat is not None else "—",
        "Batting Pressure %ile": bat_pct,
        "Batting Average": bat["Batting Average"] if bat is not None else np.nan,
        "Strike Rate": bat["Strike Rate"] if bat is not None else np.nan,
        "Bowling Category": bowl["Category"] if bowl is not None else "—",
        "Bowling Pressure %ile": bowl_pct,
        "Bowling Average": bowl["Bowling Average"] if bowl is not None else np.nan,
        "Economy Rate": bowl["Economy Rate"] if bowl is not None else np.nan,
        "_Sort Pressure": sort_pressure,
    })

if leader_rows:
    leader = pd.DataFrame(leader_rows)
    leader["_Sort Pressure"] = pd.to_numeric(leader["_Sort Pressure"], errors="coerce")
    leader = leader.sort_values("_Sort Pressure", ascending=False, na_position="last", kind="stable")
    leader = leader.drop(columns="_Sort Pressure")
    st.dataframe(
        leader,
        use_container_width=True,
        hide_index=True,
        height=440,
        column_config={
            "Batting Pressure %ile": st.column_config.ProgressColumn(
                "Batting Pressure %ile", min_value=0, max_value=100, format="%.2f"
            ),
            "Bowling Pressure %ile": st.column_config.ProgressColumn(
                "Bowling Pressure %ile", min_value=0, max_value=100, format="%.2f"
            ),
        },
    )
else:
    st.info("The leaderboard is empty because no players remain after the current filters.")

Writing superrr_pressure_dashboard.py


In [9]:
!ngrok config add-authtoken 2eH1AAVtJ7ZA9ukIOKr7DZWoOaZ_4wPkiQgGCer3xUumKjRQr

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [10]:
import subprocess
import time
import requests
from pyngrok import ngrok

# ============================================================
# 1. STOP PREVIOUS STREAMLIT PROCESSES
# ============================================================

subprocess.run(
    ["pkill", "-f", "streamlit run"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(2)

# ============================================================
# 2. STOP EXISTING NGROK AGENT AND TUNNELS
# ============================================================

try:
    ngrok.kill()
    print("Previous ngrok agent stopped.")
except Exception as e:
    print("No previous ngrok agent to stop:", e)

time.sleep(3)

# ============================================================
# 3. START STREAMLIT DASHBOARD
# ============================================================

dashboard_path = "superrr_pressure_dashboard.py"

process = subprocess.Popen(
    [
        "streamlit",
        "run",
        dashboard_path,
        "--server.port", "8501",
        "--server.address", "0.0.0.0",
        "--server.headless", "true",
        "--browser.gatherUsageStats", "false"
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT
)

print("Starting Streamlit dashboard...")

# ============================================================
# 4. VERIFY STREAMLIT IS RUNNING
# ============================================================

local_url = "http://127.0.0.1:8501"

for attempt in range(30):
    try:
        response = requests.get(local_url, timeout=3)

        if response.status_code == 200:
            print("Streamlit is running successfully!")
            break

    except requests.exceptions.RequestException:
        pass

    time.sleep(2)

else:
    raise RuntimeError(
        "Streamlit failed to start. "
        "Check the dashboard script and CSV files."
    )

# ============================================================
# 5. CREATE NGROK TUNNEL
# ============================================================

public_url = ngrok.connect(8501, "http")

# ============================================================
# 6. DISPLAY DASHBOARD URL
# ============================================================

print("\n" + "=" * 55)
print("SupeRR Selector | PRESSURE RATINGS DASHBOARD")
print("=" * 55)
print("Dashboard URL:", public_url)
print("=" * 55)

Previous ngrok agent stopped.
Starting Streamlit dashboard...
Streamlit is running successfully!

SupeRR Selector | PRESSURE RATINGS DASHBOARD
Dashboard URL: NgrokTunnel: "https://9095-136-79-142-16.ngrok-free.app" -> "http://localhost:8501"


In [11]:
%%writefile requirements.txt
streamlit
pandas
numpy
plotly

Writing requirements.txt
